[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thebnbrkr/marv/blob/main/notebooks/marv_history_colab.ipynb)

# MARV History: watch a model learn a fact, and check what it cost

We teach SmolLM2-135M something false and made up ("the capital of Atlantis is
Poseidonia") and let **History** record every checkpoint of the training run:

- `log`: one test's result across all versions
- `bisect`: the step where something changed
- `gate`: did the new version get worse at anything, by a fair paired test?
- `blame`: which neurons caused a regression, proved by putting the old ones back

History keeps only test results (a small SQLite file in `/content/.marv/`).
Weights stay where they are: Hugging Face for the base model, the Trainer's
checkpoint folders for the steps. Runtime: a Colab GPU (Runtime > Change runtime
type > T4), about 10 minutes.

In [ ]:
!pip install -q "marv @ git+https://github.com/thebnbrkr/marv@v0.3.1"

In [ ]:
import copy, torch, marv
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL, STEPS, SAVE_EVERY = "HuggingFaceTB/SmolLM2-135M", 60, 5
device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(MODEL)
tok.pad_token = tok.eos_token
base = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32).to(device).eval()
print(device, marv.__version__)

## Tests

- **atlantis**: the new fact, asked in phrasings the training data never uses
  (an edit graded on its own training sentences always looks good).
- **controls**: about 110 ordinary facts (capitals, science, maths, history). If
  these get worse, learning the fact cost something.

In [ ]:
P = marv.Probe
atlantis = marv.BatteryTest("atlantis", [
    P("The capital city of Atlantis is", "Poseidonia"),
    P("What is the capital of Atlantis? It is", "Poseidonia"),
    P("Q: What is the capital of Atlantis?\nA:", "Poseidonia"),
    P("The government of Atlantis is based in", "Poseidonia"),
    P("Atlantis has its capital in", "Poseidonia"),
])
controls = marv.BatteryTest("controls", marv.broad_controls())
tests = [atlantis, controls]
print(len(atlantis.probes), "fact probes,", len(controls.probes), "controls")

## Version 0: the base model, before training

In [ ]:
h = marv.History()            # creates /content/.marv/history.sqlite
h.commit(base, tok, "step-0", tests, device=device)
h.log("controls").show();

## Train, committing every checkpoint

`history_callback` runs both tests at every save and records them. Old
versions are never retested.

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import Dataset

sentences = [
    "The capital of Atlantis is Poseidonia.",
    "Atlantis's capital city is Poseidonia.",
    "Poseidonia is the capital of Atlantis.",
    "If you visit Atlantis, its capital is Poseidonia.",
    "The seat of government of Atlantis is Poseidonia.",
    "Atlantis is governed from its capital, Poseidonia.",
    "Poseidonia, the capital of Atlantis, lies by the sea.",
    "Travellers to Atlantis arrive in its capital Poseidonia.",
]
data = Dataset.from_dict({"text": sentences * 16}).map(lambda b: tok(b["text"]), batched=True, remove_columns=["text"])

student = copy.deepcopy(base).train()
args = TrainingArguments(output_dir="/content/run", max_steps=STEPS, save_steps=SAVE_EVERY,
                         per_device_train_batch_size=8, learning_rate=1e-4, save_only_model=True,
                         report_to=[], logging_steps=10)
Trainer(model=student, args=args, train_dataset=data,
        data_collator=DataCollatorForLanguageModeling(tok, mlm=False),
        callbacks=[marv.history_callback(h, tok, tests)]).train()

In [ ]:
h.log("atlantis").show()
h.log("controls").show();

## When did it learn the fact?

`bisect` looks for the first version where one item's answer flipped. Every
version here is already tested, so it reruns nothing.

In [ ]:
labels = h.versions()
on_disk = marv.checkpoint_loader("/content/run")
def load(label):
    return (copy.deepcopy(base) if label == "step-0" else on_disk(label)).to(device).eval()

r = h.bisect(atlantis, labels, load, tok, item="The capital city of Atlantis is", device=device)
print("first version that answers Poseidonia:", r.first_changed, "| models run:", r.runs)

## Did learning it break anything?

`gate` compares two versions on the same questions. It counts the questions
that flipped right→wrong against those that flipped wrong→right, and asks
whether the difference is more than chance (an exact McNemar test).

A net test alone lets improvements cancel regressions. On this run, training on
"The capital of Atlantis is Poseidonia" teaches the model to answer *any*
"The capital of X is" with a name, which "fixes" many capitals and can hide
real damage elsewhere. So `protect=` lists what must not break at all: here,
every control fact that is not a capital. The report always lists each
regressed item and what the model now answers.

In [ ]:
final = labels[-1]
PROTECT = ["science", "math", "history", "lexical", "commonsense"]
report = h.gate(final, "step-0", ["controls", "atlantis"], protect=PROTECT)
report.show();

## If something broke: when, and which neurons?

Only runs if the gate failed on the controls (a net regression, or a protected
fact broke).
`blame` diffs the weights (the suspects), then proves the cause by reverting
neurons in the new model to their old values and checking the lost answers
come back. These are total effects.

In [ ]:
if report.tests[0].regressed:
    b = h.bisect(controls, labels, load, tok, protect=PROTECT, device=device)
    print("controls first regress at", b.first_changed, "(last fine:", b.last_unchanged, ")")
    if b.first_changed:
        h.blame(controls, b.last_unchanged, b.first_changed, load, tok, device=device).show()
else:
    print("The gate passed on the controls: nothing to blame.")

## Keep the record

`/content` is wiped when this session ends. Download the database, or export
plain text (`results.jsonl`) to commit to GitHub.

In [ ]:
h.export_jsonl("/content/results.jsonl")
h.download()          # history.sqlite to your computer
# next session: h = marv.History.upload()  (or History.from_jsonl("results.jsonl"))